In [1]:
import sys
from pathlib import Path

# Add the parent directory of the package to sys.path
sys.path.append(str(Path.cwd().parent))

# from inciweb_timeseries_scraper import get_all_data
import argparse
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import csv
import sys


In [ ]:
# import feedparser

# # Replace with the URL of the RSS feed you want to read
# rss_url = "https://inciweb.wildfire.gov/incidents/rss.xml" 
# feed = feedparser.parse(rss_url)
# records = [dict(entry) for entry in feed.entries]
# df = pd.json_normalize(feed.entries, sep='.')
# df["link_alt"] = df.get("links", []).apply(lambda L: next((x.get("href") for x in (L or []) if isinstance(x,dict) and x.get("rel")=="alternate"), None))
# df["authors_names"] = df.get("authors", []).apply(lambda A: ", ".join([a.get("name","") for a in (A or [])]) if isinstance(A,list) else None)
# df = df.drop(columns=[c for c in ["links","title_detail","summary_detail","author_detail","authors","published_parsed"] if c in df.columns])


In [5]:
import json
import time
from datetime import date
from dateutil.relativedelta import relativedelta
from googlenewsdecoder import gnewsdecoder
from gnews import GNews

fire_terms = [
    "wildfire",
    "forest fire",
    "bushfire",
    "brush fire",
    "fire season",
    "wildland fire"
]

queries = [
    "wildfire",
    "forest fire",
    "bushfire",
    "brush fire",
    "fire season",
    "wildland fire"

]




def get_article_data(article_url, google_news):

    interval_time = 1

    try:

        decoded_url = gnewsdecoder(article_url, interval=interval_time)

        if decoded_url.get("status"):

            article = google_news.get_full_article(decoded_url["decoded_url"])

            text = (article.text or "").lower()

            if any(term in text for term in fire_terms) and len(text) >= 300:

                return {
                    "decoded_url": decoded_url["decoded_url"],
                    "title": article.title,
                    "authors": article.authors,
                    "published_date": str(article.publish_date) if article.publish_date else None,
                    "summary": article.summary,
                    "keywords": article.keywords,
                    "text": article.text
                }

            else:

                return {
                    "error": "Filtered out: not wildfire or too short",
                    "original_url": decoded_url["decoded_url"]
                }

        else:

            return {"error": decoded_url["message"], "original_url": article_url}

    except Exception as e:

        return {"error": str(e), "original_url": article_url}


def month_iter(year):

    d = date(year, 1, 1)
    end = date(year, 12, 31)

    while d <= end:

        start_tuple = (d.year, d.month, 1)

        next_month = d + relativedelta(months=1)

        end_of_month = next_month - relativedelta(days=1)

        end_tuple = (end_of_month.year, end_of_month.month, end_of_month.day)

        yield start_tuple, end_tuple

        d = next_month


if __name__ == "__main__":

    years = [ 2025]

    for year in years:

        articles_data = []
        seen = set()
        request_count = 0

        for start_tuple, end_tuple in month_iter(year):

            google_news = GNews(
                language='en',
                country='US',
                start_date=start_tuple,
                end_date=end_tuple,
                max_results=100
            )

            wf_news = []

            for q in queries:
                wf_news.extend(google_news.get_news(q))

            for item in wf_news:

                rec = get_article_data(item["url"], google_news)

                final_url = rec.get("decoded_url") or rec.get("original_url")

                if final_url and final_url in seen:
                    continue

                seen.add(final_url)

                articles_data.append(rec)

                request_count += 1

                time.sleep(0.5)

                if request_count % 6000 == 0:

                    print(f"Processed {request_count} requests. Waiting 1 hour...")
                    time.sleep(3600)

        with open(f"wildfire_articles_{year}.json", "w", encoding="utf-8") as f:

            json.dump(articles_data, f, ensure_ascii=False, indent=2)

        print(f"Year {year} DONE with length {len(articles_data)}")

        print(json.dumps(articles_data[:2], indent=2))

An error occurred while fetching the article: Article `download()` failed with 403 Client Error: Forbidden for url: https://news.stanford.edu/stories/2025/01/assessing-wildfire-health-risks on URL https://news.stanford.edu/stories/2025/01/assessing-wildfire-health-risks
An error occurred while fetching the article: Article `download()` failed with 403 Client Error: Forbidden for url: https://hsph.harvard.edu/environmental-health/news/health-impacts-of-wildfires-frequently-asked-questions/ on URL https://hsph.harvard.edu/environmental-health/news/health-impacts-of-wildfires-frequently-asked-questions/
An error occurred while fetching the article: Article `download()` failed with 403 Client Error: Forbidden for url: https://www.avma.org/blog/help-wildfire-victims-california on URL https://www.avma.org/blog/help-wildfire-victims-california
An error occurred while fetching the article: Article `download()` failed with 403 Client Error: Forbidden for url: https://hsph.harvard.edu/news/the-h

In [7]:
len(articles_data)

6074

In [10]:
files=['wildfire_articles_2023.json','wildfire_articles_2024.json','wildfire_articles_2025.json']
fields_to_keep = ['decoded_url','title', "published_date", "text"]
merged_data = []
for file in files:
    with open(file, 'r') as f:
        data = json.load(f)
        if not isinstance(data, list):
            data = [data]
            
        for entry in data:
            filtered_entry = {k: v for k, v in entry.items() if k in fields_to_keep}
            merged_data.append(filtered_entry)

# Write to a new file
with open('final_wildfire_articles.json', 'w') as f:
    json.dump(merged_data, f, indent=4)

In [12]:
j=json.load(open('final_wildfire_articles.json'))
len(j)

17037

In [ ]:
df=pd.read_json('final_wildfire_articles.json')
REPUTABLE_DOMAINS = [
    # Wire Services (gold standard — primary sources)
    "apnews.com",
    "reuters.com",
    "afp.com",
    "upi.com",

    # National U.S. Newspapers
    "nytimes.com",
    "washingtonpost.com",
    "wsj.com",
    "usatoday.com",
    "npr.org",

    # Major Regional Newspapers
    "chicagotribune.com",
    "latimes.com",
    "bostonglobe.com",
    "dallasnews.com",
    "sfchronicle.com",
    "denverpost.com",
    "seattletimes.com",
    "startribune.com",
    "tampabay.com",
    "miamiherald.com",
    "houstonchronicle.com",
    "inquirer.com",
    "baltimoresun.com",
    "sandiegouniontribune.com",
    "oregonlive.com",

    # International
    "theguardian.com",
    "ft.com",
    "economist.com",
    "abc.net.au",
    "cbc.ca",

    # Investigative / Long-form
    "propublica.org",
    "theatlantic.com",

    # Fire / Disaster Specific
    "wildfiretoday.com",

    # Government
    ".gov",
    ".org"
]
df=df[~df['decoded_url'].isna()]  # Check for NaN values in 'decoded_url' column
def is_reputable(url):
    return any(domain in url for domain in REPUTABLE_DOMAINS)
df = df[df['decoded_url'].apply(is_reputable)]
# df=df.reset_index(drop=True, inplace=True) 
df.to_csv("final_wildfire_articles.csv", index=False)